# Grammar Scoring Engine for Spoken English
### SHL Hiring Assessment 2026 — Research Engineer

**Task:** given a 45–60 s spoken audio clip, predict a continuous grammar score (0–5) matching human MOS Likert ratings.
**Metrics:** RMSE (leaderboard) and Pearson correlation.

**Final approach (one sentence):** extract layer-selected representations from three pretrained speech models (WavLM-base+, WavLM-large, Whisper-medium encoder) plus two text views of a Whisper transcript, fit one small regressor per view, and combine them with a non-negative linear blend learned from out-of-fold predictions.

| Result | Value |
|---|---|
| Out-of-fold RMSE (5-fold, real responses) | **0.497** |
| Out-of-fold Pearson (5-fold, real responses) | **0.872** |
| Public leaderboard RMSE (best) | **0.376** |
| Training RMSE | printed in Section 7 |

The written report (approach, preprocessing, architecture, evaluation, findings) is at the end of this notebook.

**Runtime:** ~2 h on a Kaggle T4 from scratch (GPU + Internet must be ON). Every expensive step caches its output; if the cache files are attached as a Kaggle dataset, the whole notebook runs in ~15 min.

## 1. Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "openai-whisper", "language_tool_python", "sentence-transformers"], check=True)

import glob, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torchaudio, nltk
from scipy.stats import pearsonr
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

## 2. Data and caching

Train and test files are concatenated in a **fixed order (all train, then all test)** and every feature matrix keeps that order.
Train and test folders reuse the same filenames (`audio_49.wav` exists in both, with different audio), so nothing is ever merged by filename.

Expensive outputs are cached under `/kaggle/working`. `cache_path()` also looks inside any attached Kaggle dataset, so previously computed files can be reused.

In [ ]:
BASE = Path("/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final")
if not BASE.exists():                                  # other mount layouts
    BASE = Path(glob.glob("/kaggle/input/**/Dataset_Final", recursive=True)[0])
TRAIN_DIR, TEST_DIR = BASE / "train", BASE / "test"
WORK = Path("/kaggle/working")

def cache_path(name):
    """Existing cache file: /kaggle/working first, then any attached Kaggle dataset."""
    p = WORK / name
    if p.exists():
        return p
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return Path(hits[0]) if hits else None

train_df = pd.read_csv(BASE / "train.csv")
test_df  = pd.read_csv(BASE / "test.csv")
paths = [str(TRAIN_DIR / f) for f in train_df.filename] + \
        [str(TEST_DIR / f) for f in test_df.filename]
n_tr  = len(train_df)
y_all = train_df["label"].values

print(f"Train: {n_tr} clips | Test: {len(test_df)} clips")
print("\nLabel counts:")
print(train_df["label"].value_counts().sort_index().to_string())

## 3. Exploratory analysis

Key observation: all **37 clips labelled 0** are exactly the contiguous block `audio_5037`–`audio_5073`. Their transcripts are empty or incoherent/off-topic, so "0" means *unscorable response*, not "very poor grammar" (there are almost no 1.0–1.5 labels). The test set (files `audio_0`–`audio_215`) contains no such block — a model trained on all 769 clips scored no test clip below 1.8.

So the model is **trained and evaluated on the 732 real responses only**, which matches the test distribution.

In [ ]:
zero = train_df.loc[train_df.label == 0, "filename"]
zero_ids = zero.str.extract(r"audio_(\d+)")[0].astype(int)
print(f"0-labelled clips: {len(zero)}  (file ids {zero_ids.min()}–{zero_ids.max()}, "
      f"contiguous: {zero_ids.max() - zero_ids.min() + 1 == len(zero)})")

durations = []
try:
    import soundfile as sf
    durations = [sf.info(p).duration for p in paths]
except Exception as e:
    print("Duration check skipped:", e)

fig, axes = plt.subplots(1, 2 if durations else 1, figsize=(12 if durations else 6, 4))
axes = np.atleast_1d(axes)
counts = train_df["label"].value_counts().sort_index()
axes[0].bar(counts.index.astype(str), counts.values, color="#4C72B0")
axes[0].set_title("Training label distribution")
axes[0].set_xlabel("Grammar score"); axes[0].set_ylabel("Clips")
if durations:
    axes[1].hist(durations, bins=30, color="#55A868")
    axes[1].set_title("Clip duration (train + test)")
    axes[1].set_xlabel("Seconds"); axes[1].set_ylabel("Clips")
plt.tight_layout(); plt.savefig("eda.png", dpi=150); plt.show()

## 4. Transcription (Whisper-small)

Each clip is transcribed with Whisper-small. `condition_on_previous_text=False` prevents repetition loops on hesitant speech.
Progress is checkpointed every 50 files so an interrupted run resumes where it stopped.

In [ ]:
TRANS = "transcriptions.csv"
if cache_path(TRANS):
    trans = pd.read_csv(cache_path(TRANS))
    print("Loaded cached transcripts")
else:
    import whisper
    partial = WORK / "transcriptions_partial.csv"
    done = pd.read_csv(partial).to_dict("records") if partial.exists() else []
    splits = ["train"] * n_tr + ["test"] * len(test_df)
    names  = list(train_df.filename) + list(test_df.filename)
    asr = whisper.load_model("small", device=DEVICE)
    for i in range(len(done), len(paths)):
        try:
            text = asr.transcribe(paths[i], language="en", fp16=(DEVICE == "cuda"),
                                  condition_on_previous_text=False)["text"].strip()
        except Exception:
            text = ""
        done.append({"filename": names[i], "split": splits[i], "transcript": text})
        if (i + 1) % 50 == 0:
            pd.DataFrame(done).to_csv(partial, index=False)
            print(f"  {i+1}/{len(paths)}")
    trans = pd.DataFrame(done)
    trans.to_csv(WORK / TRANS, index=False)
    del asr; torch.cuda.empty_cache()

assert len(trans) == len(paths), "transcript count must match the file list"
texts = trans["transcript"].fillna("").tolist()
print(texts[0][:200])

## 5. Text features

Two views of the transcript:

* **Grammar features (9 numbers):** word count, type–token ratio, words per sentence, filler rate; LanguageTool error rate (all rules and grammar-only rules); and a RoBERTa model fine-tuned on CoLA (grammatical acceptability) giving, per transcript, the mean and minimum acceptability probability across sentences and the share of sentences judged unacceptable.
* **Sentence embedding:** 768-dim `all-mpnet-base-v2` embedding of the whole transcript.

In [ ]:
TEXT = "text_features.npz"
if cache_path(TEXT):
    c = np.load(cache_path(TEXT))
    gram_feats, text_emb = c["gram"], c["emb"]
    print("Loaded cached text features")
else:
    import language_tool_python
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    from sentence_transformers import SentenceTransformer

    def basic(t):
        w = t.split(); n = max(len(w), 1)
        s = nltk.sent_tokenize(t) or [t]
        fillers = sum(t.lower().split().count(f) for f in ["um", "uh", "like", "hmm"])
        return [len(w), len(set(x.lower() for x in w)) / n, n / len(s), fillers / n]

    tool = language_tool_python.LanguageTool("en-US")          # one shared instance
    def lt(t):
        if not t:
            return [0, 0]
        m = tool.check(t); n = max(len(t.split()), 1)
        return [len(m) / n, sum(x.category == "GRAMMAR" for x in m) / n]

    cola_name = "textattack/roberta-base-CoLA"
    ctok = AutoTokenizer.from_pretrained(cola_name)
    cola = AutoModelForSequenceClassification.from_pretrained(cola_name).to(DEVICE).eval()
    def cola_feats(t):
        sents = [s for s in nltk.sent_tokenize(t) if len(s.split()) >= 2] or [t or "."]
        enc = ctok(sents, padding=True, truncation=True, max_length=128,
                   return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            p = torch.softmax(cola(**enc).logits, -1)[:, 1].cpu().numpy()
        return [p.mean(), p.min(), (p < 0.5).mean()]

    gram_feats = np.array([basic(t) + lt(t) + cola_feats(t) for t in texts], dtype=float)
    tool.close()
    text_emb = SentenceTransformer("all-mpnet-base-v2", device=DEVICE).encode(
        texts, batch_size=32, show_progress_bar=True)
    np.savez(WORK / TEXT, gram=gram_feats, emb=text_emb)

print("Grammar features:", gram_feats.shape, "| Text embedding:", text_emb.shape)

## 6. Audio embeddings from every layer

For each speech model, every hidden layer is pooled over time with **mean and standard deviation** (fixed-size vector per clip per layer).

| Model | Layers | Dim per layer | Notes |
|---|---|---|---|
| WavLM-base+ | 13 | 1536 | audio truncated to 55 s |
| WavLM-large | 25 | 2048 | audio truncated to 55 s |
| Whisper-medium encoder | 25 | 2048 | 30 s windows; padding frames dropped before pooling |

Each run takes 10–30 min on a T4 and is cached.

In [ ]:
from transformers import AutoFeatureExtractor, WavLMModel, WhisperFeatureExtractor, WhisperModel

def load_mono16k(path):
    wav, sr = torchaudio.load(path)
    wav = wav.mean(0)
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)
    return wav

def pool(hidden_states):
    """(layers, time, dim) → (layers, 2*dim): mean and std over time."""
    h = hidden_states.float()
    return torch.cat([h.mean(1), h.std(1)], -1).cpu().numpy().astype(np.float32)

def wavlm_extractor(name):
    fe  = AutoFeatureExtractor.from_pretrained(name)
    mdl = WavLMModel.from_pretrained(name).to(DEVICE).eval()
    def f(path):
        wav = load_mono16k(path)[:55 * 16000].numpy()
        inp = fe(wav, sampling_rate=16000, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            hs = mdl(**inp, output_hidden_states=True).hidden_states
        return pool(torch.stack(hs)[:, 0])
    return f, mdl

def whisper_extractor(name="openai/whisper-medium"):
    fe  = WhisperFeatureExtractor.from_pretrained(name)
    enc = WhisperModel.from_pretrained(name, torch_dtype=torch.float16).encoder.to(DEVICE).eval()
    def f(path):
        wav = load_mono16k(path).numpy()
        step = 30 * 16000
        chunks = [wav[i:i + step] for i in range(0, len(wav), step)]
        chunks = [c for c in chunks if len(c) > 16000] or [wav]   # drop <1 s leftovers
        parts = []
        for c in chunks:
            feats = fe(c, sampling_rate=16000, return_tensors="pt").input_features
            with torch.no_grad():
                hs = enc(feats.to(DEVICE, torch.float16), output_hidden_states=True).hidden_states
            n = max(2, int(len(c) / 16000 * 50))                  # real (non-padding) frames
            parts.append(torch.stack(hs)[:, 0, :n])
        return pool(torch.cat(parts, dim=1))
    return f, enc

def layer_embeddings(cache_name, make_extractor, n_layers, dim):
    if cache_path(cache_name):
        print(f"Loaded cached {cache_name}")
        return np.load(cache_path(cache_name))["layers"]
    extract, model = make_extractor()
    out = []
    for i, p in enumerate(paths):
        try:
            out.append(extract(p))
        except Exception as e:
            print("error", p, e)
            out.append(np.zeros((n_layers, dim), dtype=np.float32))
        if (i + 1) % 100 == 0:
            print(f"  {cache_name}: {i+1}/{len(paths)}")
    layers = np.stack(out)
    np.savez(WORK / cache_name, layers=layers)
    del model; torch.cuda.empty_cache()
    return layers

layers_base  = layer_embeddings("wavlm_layers.npz",
                                lambda: wavlm_extractor("microsoft/wavlm-base-plus"), 13, 1536)
layers_large = layer_embeddings("wavlm_large_layers.npz",
                                lambda: wavlm_extractor("microsoft/wavlm-large"), 25, 2048)
layers_wh    = layer_embeddings("whisper_enc_layers.npz", whisper_extractor, 25, 2048)
print(layers_base.shape, layers_large.shape, layers_wh.shape)

### 6.1 Layer probing — which layers carry grammar information?

A cross-validated Ridge regression is fit on each single layer. Error falls through the lower layers and bottoms out in the upper-middle (WavLM) or final (Whisper) layers.
The final model averages a small window around each model's best region.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

def ridge():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-1, 4, 20)))

probe = {}
for name, L in [("WavLM-base+", layers_base), ("WavLM-large", layers_large),
                ("Whisper-medium encoder", layers_wh)]:
    probe[name] = [rmse(y_all, cross_val_predict(ridge(), L[:n_tr, l], y_all, cv=kf))
                   for l in range(L.shape[1])]
    print(f"{name:24s} best layer {int(np.argmin(probe[name])):2d} "
          f"(OOF RMSE {min(probe[name]):.4f})")

plt.figure(figsize=(9, 4))
for (name, r), m in zip(probe.items(), ["s", "o", "^"]):
    plt.plot(r, marker=m, label=name)
plt.xlabel("Layer"); plt.ylabel("OOF RMSE (lower is better)")
plt.title("Grammar-score prediction error by layer")
plt.legend(); plt.tight_layout(); plt.savefig("layer_probe.png", dpi=150); plt.show()

# Layer windows used by the final model (chosen from this probe)
WINDOWS = {"WavLM-base+": (7, 12), "WavLM-large": (17, 22), "Whisper-medium encoder": (22, 25)}

## 7. Final model: per-view regressors + learned blend

* **Training set:** the 732 real responses (label > 0); the unscorable block is excluded (Section 3).
* **Regressors:** RBF-SVR (C=10) on each audio view, Ridge on the text embedding, gradient boosting on the 9 grammar features. Each is a small model chosen by out-of-fold RMSE.
* **Blend:** a `LinearRegression(positive=True)` fit on the out-of-fold predictions of the five views. Non-negative weights keep it interpretable: each weight is "how much this view is trusted".
* **Honest evaluation:** blend error is itself measured with an outer cross-validation of the blender, so no reported number uses a model that saw the row it predicts.

In [ ]:
def make(kind):
    if kind == "ridge":
        return ridge()
    if kind == "svr":
        return make_pipeline(StandardScaler(), SVR(kernel="rbf", C=10, epsilon=0.1))
    return HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05,
                                         max_iter=300, random_state=SEED)

w = WINDOWS
views = {
    "Audio: WavLM-base+ L7-11":   (layers_base[:,  slice(*w["WavLM-base+"])].mean(1), "svr"),
    "Audio: WavLM-large L17-21":  (layers_large[:, slice(*w["WavLM-large"])].mean(1), "svr"),
    "Audio: Whisper enc L22-24":  (layers_wh[:,    slice(*w["Whisper-medium encoder"])].mean(1), "svr"),
    "Text: MPNet embedding":      (text_emb, "ridge"),
    "Text: grammar features":     (gram_feats, "gbm"),
}

keep = y_all > 0
y = y_all[keep]
oof, test_pred, train_fit, rows = {}, {}, {}, []
for name, (X, kind) in views.items():
    Xtr, Xte = X[:n_tr][keep], X[n_tr:]
    oof[name] = cross_val_predict(make(kind), Xtr, y, cv=kf)
    fitted = make(kind).fit(Xtr, y)
    test_pred[name], train_fit[name] = fitted.predict(Xte), fitted.predict(Xtr)
    rows.append({"view": name, "OOF RMSE": rmse(y, oof[name]),
                 "OOF Pearson": pearsonr(y, oof[name])[0]})

O = np.column_stack(list(oof.values()))
blender = LinearRegression(positive=True).fit(O, y)
oof_blend   = np.clip(cross_val_predict(LinearRegression(positive=True), O, y, cv=kf), 0, 5)
train_blend = np.clip(blender.predict(np.column_stack(list(train_fit.values()))), 0, 5)
test_blend  = np.clip(blender.predict(np.column_stack(list(test_pred.values()))), 0, 5)

rows.append({"view": "BLEND", "OOF RMSE": rmse(y, oof_blend),
             "OOF Pearson": pearsonr(y, oof_blend)[0]})
results = pd.DataFrame(rows).set_index("view")
results["blend weight"] = list(np.round(blender.coef_, 3)) + [np.nan]
display(results.round(4))

TRAIN_RMSE = rmse(y, train_blend)
print("=" * 52)
print(f"TRAINING RMSE ({len(y)} training clips)          : {TRAIN_RMSE:.4f}")
print(f"TRAINING PEARSON                              : {pearsonr(y, train_blend)[0]:.4f}")
print(f"OOF RMSE    (5-fold, honest estimate)          : {rmse(y, oof_blend):.4f}")
print(f"OOF PEARSON (5-fold, honest estimate)          : {pearsonr(y, oof_blend)[0]:.4f}")
print("=" * 52)

## 8. Evaluation and interpretation

Training RMSE is much lower than OOF RMSE because the SVRs fit their own training points closely; the **OOF numbers are the ones to trust**.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))

jit = np.random.default_rng(SEED).uniform(-0.12, 0.12, len(y))
ax[0].scatter(y + jit, oof_blend, s=10, alpha=0.4)
ax[0].plot([1.5, 5], [1.5, 5], "r--")
ax[0].set_title(f"Out-of-fold predictions\nRMSE {rmse(y, oof_blend):.3f} · "
                f"Pearson {pearsonr(y, oof_blend)[0]:.3f}")
ax[0].set_xlabel("Human score (jittered)"); ax[0].set_ylabel("Predicted score")

per = pd.DataFrame({"label": y, "pred": oof_blend}).groupby("label")["pred"]
ax[1].bar(per.mean().index.astype(str), per.mean().values - per.mean().index.values,
          color="#C44E52")
ax[1].axhline(0, color="black", lw=0.8)
ax[1].set_title("Mean bias by human score\n(+ = over-predicted)")
ax[1].set_xlabel("Human score"); ax[1].set_ylabel("Mean (prediction − truth)")

ax[2].barh(list(oof.keys()), blender.coef_, color="#4C72B0")
ax[2].invert_yaxis()
ax[2].set_title("Blend weights (non-negative)")
ax[2].set_xlabel("Weight")

plt.tight_layout(); plt.savefig("evaluation.png", dpi=150); plt.show()

print("Per-label OOF error:")
print(pd.DataFrame({"n": per.size(), "mean pred": per.mean().round(2),
                    "RMSE": pd.Series({k: rmse(np.full(len(g), k), g)
                                       for k, g in per})}).round(3).to_string())

## 9. Submission

In [ ]:
submission = pd.DataFrame({"filename": test_df["filename"], "label": test_blend})
assert len(submission) == 216 and submission["label"].notna().all()
assert (submission["filename"].values == test_df["filename"].values).all()
submission.to_csv(WORK / "submission.csv", index=False)
print(submission["label"].describe().round(3).to_string())
submission.head()

## 10. Report

### 10.1 Approach
Grammar is judged from **how** people speak (fluency, hesitation, control) and **what** they say (sentence structure, errors). The pipeline therefore uses two kinds of evidence:

1. **Audio representations** from pretrained self-supervised / ASR speech models, taken from the layers that best predict the score (found by layer probing).
2. **Text representations** of a Whisper transcript: an explicit grammar view (LanguageTool errors, CoLA acceptability, fluency statistics) and a semantic sentence embedding.

Each view gets its own small regressor; a non-negative linear blend combines them. Everything is selected using 5-fold out-of-fold error, never training error.

### 10.2 Preprocessing
* Audio: mono, resampled to 16 kHz; WavLM inputs truncated to 55 s; Whisper encoder fed 30 s windows with padding frames removed before pooling.
* Pooling: mean + standard deviation over time for every layer (captures both typical content and variability).
* Text: Whisper-small transcription with `condition_on_previous_text=False` (avoids repetition loops).
* Labels: the 37 "0" clips (a contiguous block of unscorable recordings absent from the test set) are excluded from training and evaluation.
* Strict train/test ordering; no filename merges, since filenames collide across folders.

### 10.3 Architecture
```
            ┌─ WavLM-base+  (L7–11, mean+std) ─► SVR ─┐
audio ──────┼─ WavLM-large  (L17–21)          ─► SVR ─┤
            └─ Whisper-medium enc (L22–24)    ─► SVR ─┤
                                                      ├─► non-negative linear blend ─► score
 Whisper-small ─► transcript ─┬─ MPNet embedding ─► Ridge ─┤
                              └─ LanguageTool + CoLA + fluency (9) ─► GBM ─┘
```

### 10.4 Results and how we got there

| Step | Change | OOF RMSE | Public LB RMSE |
|---|---|---|---|
| 1 | Whisper transcript → 17 hand-crafted features → RandomForest | ~1.05 | 0.675 |
| 2 | Wav2Vec2 (last layer) → Ridge | 0.739 | 0.646 |
| 3 | + text embedding + grammar features, blended | 0.649 | 0.472 |
| 4 | WavLM-base+, **layer-probed** (best layer instead of last) | 0.558 | 0.382 |
| 5 | + WavLM-large, + Whisper encoder, SVR, two-stage zero handling | 0.496 | **0.376** |
| 6 | Exclude unscorable block, train on real responses only (this notebook) | 0.497* | 0.383 |

\* Step 6 is measured on real responses only (the test distribution); on that same measure step 5 scores 0.507, so step 6 is the better model. Public-leaderboard differences below ~0.02 are noise: it is scored on ~130 clips.

### 10.5 Findings
* **Layer choice matters more than model choice.** For WavLM the last layer is worse than the upper-middle layers (pre-training objective specialises the top). For Whisper the last encoder layers are best, because they are trained to feed a text decoder and hold the most language-level information.
* **Audio beats text on its own** (best single view ~0.53 vs ~0.75 RMSE), likely because Whisper normalises some disfluencies and errors, and raters' grammar judgements are tied to fluency. Text still earns ~30% of the blend weight: the views are complementary.
* **Remaining error is regression toward the mean:** low scores are over-predicted and high scores under-predicted. Part of this is irreducible rater noise.

### 10.6 Limitations and next steps
* The model never predicts the "unscorable" category; a production system would need a separate silence/off-topic detector (the 0-block was trivially separable from audio in our experiments).
* Next experiments: fine-tuning the best speech encoder end-to-end with a regression head; ordinal/distribution-aware losses to reduce the compression toward the mean; verbatim (disfluency-preserving) transcription.